In [0]:
STORAGE_PATH = "abfss://raw@rxmaxdetestsa.dfs.core.windows.net/personal-learning/parth/retailpulse/raw"

In [0]:
# Databricks notebook source
%pip install faker

# COMMAND ----------

import json
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker
from pathlib import Path

fake = Faker()
Faker.seed(42)
random.seed(42)

# COMMAND ----------

# CONFIG - change these
# STORAGE_PATH = "dbfs:/raw/"  # or abfss:// path
# If not using Unity Catalog volumes, use DBFS for learning:
# STORAGE_PATH = "/tmp/retailpulse/raw"

START_DATE = "2026-04-25"   # 7 days back from today-ish
DAYS_TO_GENERATE = 7
ORDERS_PER_DAY = 10000
TOTAL_CUSTOMERS = 1000
TOTAL_PRODUCTS = 500

# COMMAND ----------

# ---------- CUSTOMERS (generated once) ----------
def generate_customers(n):
    customers = []
    for i in range(n):
        c = {
            "customer_id": f"CUST_{i:05d}",
            "name": fake.name(),
            "email": fake.email(),
            "city": fake.city(),
            "country": fake.country_code(),
            "signup_date": fake.date_between(start_date="-2y", end_date="today").isoformat(),
            "loyalty_tier": random.choice(["bronze", "silver", "gold", "platinum"])
        }
        # 2% nulls in email
        if random.random() < 0.02:
            c["email"] = None
        customers.append(c)
    return customers

# ---------- PRODUCTS (generated once) ----------
def generate_products(n):
    categories = ["electronics", "apparel", "home", "books", "sports", "beauty"]
    products = []
    for i in range(n):
        products.append({
            "product_id": f"PROD_{i:05d}",
            "name": fake.catch_phrase(),
            "category": random.choice(categories),
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 500)
        })
    return products

# ---------- ORDERS (per day) ----------
def generate_orders(date_str, count, customer_ids, product_ids, day_index):
    orders = []
    for _ in range(count):
        order = {
            "order_id": str(uuid.uuid4()),
            "customer_id": random.choice(customer_ids),
            "product_id": random.choice(product_ids),
            "quantity": random.randint(1, 5),
            "order_timestamp": f"{date_str}T{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}",
            "status": random.choices(
                ["completed", "pending", "cancelled", "refunded"],
                weights=[70, 15, 10, 5]
            )[0],
            "payment_method": random.choice(["card", "upi", "wallet", "cod"])
        }
        # 2% null quantity
        if random.random() < 0.02:
            order["quantity"] = None
        # Schema drift: from day 5+, add 'discount_code' field
        if day_index >= 4:
            order["discount_code"] = random.choice([None, "SAVE10", "FLAT50", "WELCOME"])
        orders.append(order)
    
    # 1% duplicates
    dup_count = int(count * 0.01)
    orders.extend(random.sample(orders, dup_count))
    return orders

# COMMAND ----------

def write_json(records, path):
    """Write list of dicts as newline-delimited JSON"""
    dbutils.fs.mkdirs(path.rsplit("/", 1)[0])
    content = "\n".join(json.dumps(r) for r in records)
    dbutils.fs.put(path, content, overwrite=True)

# COMMAND ----------

# Generate static dimensions once
customers = generate_customers(TOTAL_CUSTOMERS)
products = generate_products(TOTAL_PRODUCTS)

customer_ids = [c["customer_id"] for c in customers]
product_ids = [p["product_id"] for p in products]

# Write customers and products (single snapshot for now)
write_json(customers, f"{STORAGE_PATH}/customers/customers.json")
write_json(products, f"{STORAGE_PATH}/products/products.json")
print(f"Wrote {len(customers)} customers, {len(products)} products")

# COMMAND ----------

# Generate 7 days of orders
start = datetime.strptime(START_DATE, "%Y-%m-%d")
for day_idx in range(DAYS_TO_GENERATE):
    date = start + timedelta(days=day_idx)
    date_str = date.strftime("%Y-%m-%d")
    orders = generate_orders(date_str, ORDERS_PER_DAY, customer_ids, product_ids, day_idx)
    path = f"{STORAGE_PATH}/orders/{date_str}/orders.json"
    write_json(orders, path)
    print(f"Day {day_idx+1} ({date_str}): {len(orders)} orders written")

print("Generation complete.")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Wrote 185961 bytes.
Wrote 63979 bytes.
Wrote 1000 customers, 500 products
Wrote 2178715 bytes.
Day 1 (2026-04-25): 10100 orders written
Wrote 2178692 bytes.
Day 2 (2026-04-26): 10100 orders written
Wrote 2178877 bytes.
Day 3 (2026-04-27): 10100 orders written
Wrote 2178468 bytes.
Day 4 (2026-04-28): 10100 orders written
Wrote 2443943 bytes.
Day 5 (2026-04-29): 10100 orders written
Wrote 2443745 bytes.
Day 6 (2026-04-30): 10100 orders written
Wrote 2443902 bytes.
Day 7 (2026-05-01): 10100 orders written
Generation complete.
